# NLP - Sentiment Analysis System
This notebook demonstrates text preprocessing, training, evaluation, and visualization for three sentiment classification models on the IMDB Movie Reviews dataset:
1. **Naïve Bayes**
2. **Logistic Regression**
3. **Bi-directional LSTM (PyTorch)**

## 1. Import Libraries

In [ ]:
import os
import re
import urllib.request
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report

## 2. Load Dataset
We will download the IMDB Movie Reviews CSV dataset if it is not already present locally.

In [ ]:
DATA_DIR = './data'
CSV_PATH = os.path.join(DATA_DIR, 'IMDB-Dataset.csv')
DATASET_URL = "https://raw.githubusercontent.com/laxmimerit/All-CSV-ML-Data-Files-Download/master/IMDB-Dataset.csv"

if not os.path.exists(CSV_PATH):
    os.makedirs(DATA_DIR, exist_ok=True)
    print("Downloading dataset (66MB)... This might take a moment.")
    urllib.request.urlretrieve(DATASET_URL, CSV_PATH)
    print("Download complete.")
else:
    print("Dataset found locally.")

# Load data
df = pd.read_csv(CSV_PATH)
print(f"Total reviews in dataset: {len(df)}")
print(df.head())

## 3. Data Preprocessing
We clean the text data, remove HTML tags, web URLs, and stopwords, and tokenize the reviews.

In [ ]:
# Define stopwords directly to keep notebook self-contained
STOPWORDS = set([
    'a', 'about', 'above', 'after', 'again', 'against', 'all', 'am', 'an', 'and', 'any', 'are', 'arent', 'as', 'at',
    'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'cant', 'cannot', 'could',
    'couldnt', 'did', 'didnt', 'do', 'does', 'doesnt', 'doing', 'dont', 'down', 'during', 'each', 'few', 'for', 'from',
    'further', 'had', 'hadnt', 'has', 'hasnt', 'have', 'havent', 'having', 'he', 'hed', 'hell', 'hes', 'her', 'here',
    'heres', 'hers', 'herself', 'him', 'himself', 'his', 'how', 'hows', 'i', 'id', 'ill', 'im', 'ive', 'if', 'in',
    'into', 'is', 'isnt', 'it', 'its', 'itself', 'lets', 'me', 'more', 'most', 'mustnt', 'my', 'myself', 'no', 'nor',
    'not', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'ought', 'our', 'ours', 'ourselves', 'out', 'over', 'own',
    'same', 'shant', 'shed', 'shell', 'shes', 'should', 'shouldnt', 'so', 'some', 'such', 'than', 'that', 'thats',
    'the', 'their', 'theirs', 'them', 'themselves', 'then', 'there', 'theres', 'these', 'they', 'theyd', 'theyll',
    'theyre', 'theyve', 'this', 'those', 'through', 'to', 'too', 'under', 'until', 'up', 'very', 'was', 'wasnt',
    'we', 'wed', 'well', 'were', 'weve', 'werent', 'what', 'whats', 'when', 'whens', 'where', 'wheres', 'which',
    'while', 'who', 'whos', 'whom', 'why', 'whys', 'with', 'wont', 'would', 'wouldnt', 'you', 'youd', 'youll',
    'youre', 'youve', 'your', 'yours', 'yourself', 'yourselves'
])

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'<[^>]+>', ' ', text)  # remove HTML
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)  # remove URLs
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)  # alphanumeric
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def tokenize(text):
    cleaned = clean_text(text)
    tokens = cleaned.split()
    return [w for w in tokens if w not in STOPWORDS]

# Sample a subset of reviews for fast compilation in notebook (e.g. 5,000 reviews)
sample_df = df.sample(n=5000, random_state=42).reset_index(drop=True)
sample_df['label'] = sample_df['sentiment'].map({'positive': 1.0, 'negative': 0.0})

train_texts, test_texts, train_labels, test_labels = train_test_split(
    sample_df['review'].tolist(), sample_df['label'].tolist(), test_size=0.2, random_state=42
)

print(f"Training set size: {len(train_texts)}")
print(f"Test set size: {len(test_texts)}")

## 4. Train Traditional Machine Learning Models (TF-IDF)
We fit a TF-IDF vectorizer and train Naïve Bayes and Logistic Regression classifiers.

In [ ]:
# TF-IDF Transformation
vectorizer = TfidfVectorizer(max_features=5000, preprocessor=clean_text, tokenizer=tokenize, token_pattern=None)
X_train_tfidf = vectorizer.fit_transform(train_texts)
X_test_tfidf = vectorizer.transform(test_texts)

# 1. Naïve Bayes
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, train_labels)
nb_preds = nb_model.predict(X_test_tfidf)
print("=== Naïve Bayes Classification Report ===")
print(classification_report(test_labels, nb_preds, target_names=['negative', 'positive']))

# 2. Logistic Regression
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_tfidf, train_labels)
lr_preds = lr_model.predict(X_test_tfidf)
print("=== Logistic Regression Classification Report ===")
print(classification_report(test_labels, lr_preds, target_names=['negative', 'positive']))

## 5. Train Deep Learning Model: LSTM (PyTorch)
We create a Vocabulary, build a padded text Dataset, define the Bidirectional LSTM model, and write the training loop.

In [ ]:
# Build Vocabulary
word_counts = {}
for text in train_texts:
    for token in tokenize(text):
        word_counts[token] = word_counts.get(token, 0) + 1

sorted_words = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)
word2idx = {"<PAD>": 0, "<UNK>": 1}
for idx, (word, _) in enumerate(sorted_words[:9998], start=2): # Max vocab size = 10,000
    word2idx[word] = idx

def text_to_indices(text, max_len=100):
    tokens = tokenize(text)
    indices = [word2idx.get(w, 1) for w in tokens]
    if len(indices) < max_len:
        indices += [0] * (max_len - len(indices))
    else:
        indices = indices[:max_len]
    return indices

class SentimentDataset(Dataset):
    def __init__(self, texts, labels, max_len=100):
        self.labels = labels
        self.data = [text_to_indices(text, max_len) for text in texts]
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return torch.tensor(self.data[idx], dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.float)

train_dataset = SentimentDataset(train_texts, train_labels)
test_dataset = SentimentDataset(test_texts, test_labels)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# Define LSTM Model
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=128, output_dim=1, num_layers=1, bidirectional=True):
        super(LSTMClassifier, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers=num_layers, 
                            bidirectional=bidirectional, batch_first=True)
        fc_in = hidden_dim * 2 if bidirectional else hidden_dim
        self.fc = nn.Linear(fc_in, output_dim)
        self.dropout = nn.Dropout(0.5)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, text):
        embedded = self.dropout(self.embedding(text))
        lstm_out, (hidden, cell) = self.lstm(embedded)
        if self.lstm.bidirectional:
            hidden_last = torch.cat((hidden[-2, :, :], hidden[-1, :, :]), dim=1)
        else:
            hidden_last = hidden[-1, :, :]
        out = self.fc(self.dropout(hidden_last))
        return self.sigmoid(out)

### Training Loop for PyTorch LSTM

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

lstm_model = LSTMClassifier(vocab_size=len(word2idx))
lstm_model.to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(lstm_model.parameters(), lr=0.001)

epochs = 8
loss_history = []

for epoch in range(epochs):
    lstm_model.train()
    epoch_loss = 0
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = lstm_model(inputs).squeeze()
        if outputs.dim() == 0: outputs = outputs.unsqueeze(0)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        
    avg_loss = epoch_loss / len(train_loader)
    loss_history.append(avg_loss)
    print(f"Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f}")

### Evaluate LSTM on Test Set

In [ ]:
lstm_model.eval()
lstm_preds = []
with torch.no_grad():
    for inputs, targets in test_loader:
        inputs = inputs.to(device)
        outputs = lstm_model(inputs).squeeze()
        if outputs.dim() == 0: outputs = outputs.unsqueeze(0)
        preds = (outputs >= 0.5).float().cpu().numpy().tolist()
        lstm_preds.extend(preds)

print("=== LSTM Classification Report ===")
print(classification_report(test_labels, lstm_preds, target_names=['negative', 'positive']))

## 6. Visualizations
Here we plot accuracy comparisons, the LSTM loss curve, and the confusion matrices for all three models.

In [ ]:
# Calculate Accuracies
nb_acc = accuracy_score(test_labels, nb_preds)
lr_acc = accuracy_score(test_labels, lr_preds)
lstm_acc = accuracy_score(test_labels, lstm_preds)

sns.set_theme(style="whitegrid")
plt.figure(figsize=(15, 4))

# Plot 1: Model Accuracies
plt.subplot(1, 2, 1)
models = ['Naïve Bayes', 'Logistic Regression', 'LSTM (PyTorch)']
accuracies = [nb_acc * 100, lr_acc * 100, lstm_acc * 100]
sns.barplot(x=models, y=accuracies, palette='viridis')
plt.ylabel('Accuracy (%)')
plt.ylim(50, 100)
plt.title('Validation Accuracy Comparison')
for i, v in enumerate(accuracies):
    plt.text(i, v + 1, f"{v:.1f}%", ha='center', fontweight='bold')

# Plot 2: LSTM Training Loss Curve
plt.subplot(1, 2, 2)
plt.plot(range(1, epochs + 1), loss_history, marker='o', color='#10b981', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Binary Cross-Entropy Loss')
plt.title('PyTorch LSTM Training Loss Curve')
plt.xticks(range(1, epochs + 1))

plt.tight_layout()
plt.show()

### Confusion Matrices Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

cms = [
    (confusion_matrix(test_labels, nb_preds), "Naïve Bayes"),
    (confusion_matrix(test_labels, lr_preds), "Logistic Regression"),
    (confusion_matrix(test_labels, lstm_preds), "LSTM (PyTorch)")
]

for idx, (cm, name) in enumerate(cms):
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[idx],
                xticklabels=['Negative', 'Positive'], 
                yticklabels=['Negative', 'Positive'])
    axes[idx].set_title(f"{name} Confusion Matrix")
    axes[idx].set_xlabel('Predicted Label')
    axes[idx].set_ylabel('True Label')

plt.tight_layout()
plt.show()